# Piano lab: the engine's ideas, one at a time

Six small experiments, each one idea, each one you can hear. Together they are the core
of the engine; at the end of each step a pointer shows where the same idea lives in the
code. You need only numpy, scipy and matplotlib. Run the cells in order (Shift+Enter);
change the numbers and run them again, that is the point.

| Step | Idea | Radio counterpart |
|---|---|---|
| 1 | One partial: a decaying phasor | an I/Q oscillator ringing down, Q |
| 2 | A note: many partials, stretched | harmonics, dispersion |
| 3 | The hammer: a pulse and its spectrum | pulse width vs bandwidth |
| 4 | Two strings, one bridge | coupled tanks, mode splitting |
| 5 | The quill: where you pluck | a comb filter |
| 6 | The space: impulse responses | multipath, channel response |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.signal import fftconvolve

SR = 48000  # samples per second, as the plugin usually runs

def play(y, label=""):
    # normalized to −3 dBFS so every example plays at a similar loudness
    y = np.asarray(y, float)
    y = 0.7 * y / (np.max(np.abs(y)) + 1e-12)
    if label:
        print(label)
    display(Audio(y, rate=SR, normalize=False))

def db(x):
    return 20 * np.log10(np.abs(x) + 1e-12)

def spectrum(y, fmax=5000, label=None, ax=None):
    # magnitude spectrum in dB (Hann window)
    w = np.hanning(len(y))
    Y = np.fft.rfft(y * w)
    f = np.fft.rfftfreq(len(y), 1 / SR)
    ax = ax or plt.gca()
    keep = f <= fmax
    ax.plot(f[keep], db(Y[keep]) - db(Y).max(), lw=0.8, label=label)
    ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("dB"); ax.set_ylim(-90, 3)

## 1 · One partial: a decaying phasor

A string's partial is a resonance that rings down, like a tank circuit after a pulse.
Write it as a **phasor**: a pointer of length *a* that turns by 2π·f/SR each sample and
shrinks a little each sample. In complex numbers that is one multiplication per sample:

$$z[n] = p \cdot z[n-1], \qquad p = e^{(-1/\tau \,+\, 2\pi i f)/SR}$$

The sound is the real part of *z* (the I channel); the imaginary part is Q. τ is the time
the amplitude takes to fall by 1/e, and the resonator's quality factor is **Q = π·f·τ**.

In [ ]:
f, tau = 262.0, 1.5            # middle C, 1.5 s amplitude decay
p = np.exp((-1 / tau + 2j * np.pi * f) / SR)
print(f"|p| = {abs(p):.8f} (shrink per sample),  angle = {np.angle(p):.5f} rad,  Q = {np.pi * f * tau:.0f}")

n = int(3 * SR)
z = np.empty(n, complex)
z[0] = 1.0
for k in range(1, n):          # the recursion: what the engine's resonator banks run
    z[k] = p * z[k - 1]

direct = np.exp((-1 / tau + 2j * np.pi * f) * np.arange(n) / SR)   # the closed form
print("recursion equals closed form:", np.allclose(z, direct))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
k = np.arange(400)
ax[0].plot(z.real[k], z.imag[k], lw=0.8); ax[0].set_aspect("equal")
ax[0].set_title("I/Q: the first 400 samples (barely shrinking yet)")
t = np.arange(n) / SR
ax[1].plot(t, z.real, lw=0.3); ax[1].plot(t, np.exp(-t / tau), "r--", lw=1)
ax[1].set_title("the real part rings down along e^(−t/τ)"); ax[1].set_xlabel("s")
plt.tight_layout(); plt.show()
play(z.real, "a single partial: pure, a bit like a tuning fork")

**Try:** τ = 0.1 (a dull thud), τ = 10 (a bell). The engine runs one such resonator per
partial, per string, per polarization, thousands at once.

**In the engine:** `engine/src/kernel.rs` (banks of complex one-poles, side by side so the
CPU can do eight at a time) and `engine/src/voice.rs` (`Re(w·e^{λ m dt})` in its header is
exactly this). The Python version: `proto/string_model.py`.

## 2 · A note: many partials, stretched

A string vibrates at its fundamental and at multiples of it. A real piano string is
**stiff**, and stiffness pushes the upper partials sharp:

$$f_n = n \, f_0 \sqrt{1 + B n^2}$$

B is the **inharmonicity**, around 10⁻⁴…10⁻³ in the middle of a piano, more in the bass
and the top. Higher partials also die faster (more loss per cycle). Same resonator as in
step 1, just many of them added up.

In [ ]:
def note(f0, B, n_partials=40, dur=3.0, amp=None, tau1=3.0):
    n = np.arange(1, n_partials + 1)
    fn = n * f0 * np.sqrt(1 + B * n**2)
    fn = fn[fn < SR / 2 * 0.9]                         # nothing above the Nyquist limit
    n = n[: len(fn)]
    a = (1.0 / n) if amp is None else amp(n, fn)        # level per partial
    tau = tau1 / (1 + 0.02 * n**2)                      # upper partials decay faster
    t = np.arange(int(dur * SR)) / SR
    y = sum(ai * np.exp(-t / ti) * np.cos(2 * np.pi * fi * t) for ai, ti, fi in zip(a, tau, fn))
    return y, fn

f0 = 65.4  # C2, in the bass, where stiffness is easy to hear
y0, f_harm = note(f0, B=0.0)
y1, f_stiff = note(f0, B=4e-4)

fig, ax = plt.subplots(figsize=(11, 3.5))
spectrum(y1[: SR], fmax=2500, label="stiff string, B = 4e-4", ax=ax)
for k in range(1, 38):
    ax.axvline(k * f0, color="gray", lw=0.4, alpha=0.5)
ax.set_title("gray lines: exact multiples of f0; the stiff string's partials run ahead of them")
ax.legend(); plt.show()
print("partial 20: harmonic {:.0f} Hz, stiff {:.0f} Hz ({:+.0f} cents)".format(
    f_harm[19], f_stiff[19], 1200 * np.log2(f_stiff[19] / f_harm[19])))
play(y0, "B = 0: a perfectly harmonic string (organ-like, a little synthetic)")
play(y1, "B = 4e-4: a stiff string (more piano-like, slightly metallic)")

**Try:** B = 2e-3, then 1e-2 (a thick, short string: bell-like). This stretch is why
piano tuners tune octaves slightly wide.

**In the engine:** `engine/src/physics.rs` turns string length, tension, mass and
stiffness into the partials' frequencies and losses; `proto/scale.py` holds those values
across the keyboard.

## 3 · The hammer: a pulse and its spectrum

The hammer hits the string for a few milliseconds. A short pulse has a wide spectrum, a
long pulse a narrow one, exactly as with a keyed carrier: hard keying clicks, soft
keying does not. The felt is a nonlinear spring, so a harder blow squeezes it more and
the contact gets **shorter**: louder notes are also brighter.

Here the force is a half-sine of duration T, and each partial gets the pulse's spectrum
at its own frequency.

In [ ]:
def pulse_spectrum(f, T):
    # magnitude spectrum of a half-sine pulse of duration T (closed form)
    x = f * T
    return np.abs(np.cos(np.pi * x) / (1 - 4 * x**2 + 1e-9))

f0 = 262.0
fig, ax = plt.subplots(figsize=(11, 3.5))
ff = np.linspace(1, 8000, 4000)
for T, lab in [(4e-3, "soft blow: 4 ms contact"), (1e-3, "hard blow: 1 ms contact")]:
    ax.plot(ff, db(pulse_spectrum(ff, T)), label=lab)
ax.set_ylim(-60, 3); ax.set_xlabel("Hz"); ax.set_ylabel("dB"); ax.legend()
ax.set_title("shorter pulse, wider spectrum"); plt.show()

for T, lab in [(4e-3, "soft (4 ms)"), (1e-3, "hard (1 ms)")]:
    y, _ = note(f0, B=3e-4, amp=lambda n, fn, T=T: pulse_spectrum(fn, T) / n)
    play(y, f"middle C, {lab}")

In the real engine the two hardness levels sound different in level too; here every
example is normalized, so you hear only the color.

**In the engine:** `engine/src/voice.rs` simulates the contact sample by sample (the felt's
force grows faster than its compression); `proto/fit_hammer.py` fits the felt to how the
recordings change from soft to loud. The *Hammer hardness* control scales the felt.

## 4 · Two strings, one bridge

Most piano notes have two or three strings, tuned a few cents apart. They do not ring
independently: they all push on the same bridge, and the bridge pushes back. That is two
resonators coupled through a shared lossy element, like two tanks loaded by one resistor.

The math: each string is one resonator as in step 1, and the bridge adds a loss that both
share. In matrix form the system has two **modes**:

- strings moving **in phase**: both push the bridge together, energy leaves fast;
- strings moving **against each other**: their pushes cancel at the bridge, energy stays.

The hammer starts both in phase, so the sound first decays fast, and the small detuning
slowly turns energy into the slow mode: a loud attack, then a long, quieter singing.
That is the piano's **two-stage decay**. (Weinreich described it in 1977.)

In [ ]:
def two_strings(f0, cents, g, tau=20.0, dur=8.0):
    # two strings (detuned by ±cents/2) coupled through a resistive bridge of strength g
    w = 2 * np.pi * f0 * 2 ** (np.array([-1, 1]) * cents / 2400)
    A = np.diag(-1 / tau + 1j * w) - g * np.ones((2, 2))   # d/dt z = A z
    lam, V = np.linalg.eig(A)                                # the two coupled modes
    c = np.linalg.solve(V, np.array([1, 1], complex))        # struck together, in phase
    t = np.arange(int(dur * SR)) / SR
    z = (V * c) @ np.exp(np.outer(lam, t))                   # both strings over time
    bridge = (z[0] + z[1]).real                              # what the bridge, and we, hear
    return t, bridge, lam

f0, g = 262.0, 2.0
fig, ax = plt.subplots(figsize=(11, 3.8))
for cents in [0.0, 0.5, 2.0]:
    t, y, lam = two_strings(f0, cents, g)
    env = np.sqrt(np.convolve(y**2, np.ones(2400) / 2400, "same"))
    ax.plot(t, db(env) - db(env).max(), label=f"detuned {cents} cents")
    print(f"{cents} cents: mode decay times {np.round(-1 / lam.real, 2)} s")
ax.set_ylim(-80, 3); ax.set_xlabel("s"); ax.set_ylabel("level (dB)"); ax.legend()
ax.set_title("in tune: fast decay only; slightly detuned: a fast drop, then a slow tail (with beats)")
plt.show()
for cents in [0.0, 2.0]:
    _, y, _ = two_strings(f0, cents, g)
    play(y, f"two strings, {cents} cents apart")

**Try:** g = 0 (no bridge: just beating), 5 cents (a honky-tonk). The *Unison width*
control and the *unisons* wear move exactly this detuning.

**In the engine:** `engine/src/physics.rs` builds the coupled eigenmodes per note (the
`eig` above, for 2–3 strings and two planes of motion); `engine/src/voice.rs` runs one
resonator per mode. `proto/fit_coupling.py` fits g and the losses to the recordings'
decay curves.

## 5 · The quill: where you pluck

A harpsichord string is plucked, not struck. Where it is plucked decides which partials
it can have: plucked at 1/7 of its length, the 7th, 14th, 21st partial are missing,
because the string cannot have a node exactly where it was pulled. The levels follow

$$a_n \propto \frac{\sin(n \pi \beta)}{n^2}$$

with β the pluck point as a fraction of the length. A comb filter, made of a string.

In [ ]:
f0, beta = 175.0, 1 / 7
amp = lambda n, fn: np.abs(np.sin(n * np.pi * beta)) / n**2 * n   # ×n: the force on the bridge grows with frequency
y, fn = note(f0, B=5e-5, n_partials=60, amp=amp, tau1=4.0)
fig, ax = plt.subplots(figsize=(11, 3.5))
spectrum(y[: SR], fmax=6000, ax=ax)
for k in (7, 14, 21, 28):
    ax.axvline(k * f0, color="red", lw=0.8, ls="--")
ax.set_title("plucked at 1/7: the partials at the red lines are missing"); plt.show()
play(y, "a plucked string at 1/7")
y2, _ = note(f0, B=5e-5, n_partials=60, amp=lambda n, fn: np.abs(np.sin(n * np.pi * 0.5)) / n, tau1=4.0)
play(y2, "the same string plucked in the middle: only odd partials (hollow, clarinet-like)")

**In the engine:** `proto/fit_pluck.py` finds each harpsichord note's pluck point from the
notch pattern in its recording; the *Quill brightness* control shapes the pluck itself.

## 6 · The space: impulse responses

Everything after the strings is **linear**: the soundboard, the microphone's paths, the
room. A linear system is fully described by its **impulse response**: what comes out
when you put a click in. To hear any signal through it, **convolve** the signal with that
response. In radio terms it is the channel model, and multipath is its simplest case.

First the simplest channel: the direct sound plus one floor reflection, 2 ms later and a
little weaker. Then a room: thousands of reflections, modeled as noise decaying over a
second and a half.

In [ ]:
y, _ = note(262.0, B=3e-4, dur=2.5)

# one reflection: direct + 0.7 × delayed by 2 ms (a path 0.69 m longer)
d = int(0.002 * SR)
h_floor = np.zeros(d + 1); h_floor[0] = 1.0; h_floor[d] = 0.7
H = np.fft.rfft(h_floor, 8192); fH = np.fft.rfftfreq(8192, 1 / SR)

# a room: noise decaying 60 dB in 1.5 s, after a direct click
rng = np.random.default_rng(1)
t = np.arange(int(1.5 * SR)) / SR
h_room = rng.standard_normal(len(t)) * 10 ** (-3 * t / 1.5) * 0.03
h_room[0] = 1.0

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(fH, db(H)); ax[0].set_xlim(0, 3000); ax[0].set_ylim(-20, 8)
ax[0].set_title("direct + one reflection: a comb filter (notches every 500 Hz)"); ax[0].set_xlabel("Hz")
ax[1].plot(t, h_room, lw=0.3); ax[1].set_title("a room's impulse response"); ax[1].set_xlabel("s")
plt.tight_layout(); plt.show()

play(y, "dry")
play(fftconvolve(y, h_floor), "with one floor reflection (a slightly hollow color)")
play(fftconvolve(y, h_room), "in a room")

**Try:** a longer room (3 s), or move the reflection to 5 ms. The notches move: that is
why the microphone position changes the color.

**In the engine:** `engine/src/convolution.rs` does this convolution in real time, cut
into blocks so the delay stays at 64 samples; `engine/src/mic.rs` builds the paths
(direct, floor, lid) from the microphone position; the room and the board's responses
come from `proto/soundboard.py` and `engine/src/noise.rs`.

## Where to go next

You now have all of the core: resonators (steps 1–2), how they are started (3, 5), how
they couple (4) and how the result reaches a microphone (6). What the engine adds on top
is mostly *measurement* (fitting these numbers per note from recordings) and *speed*
(thousands of resonators in real time). A good next read is `proto/string_model.py`: it
is steps 1–4 for one real note, with the real numbers.